### Structured Output

By default, LLMs return plain text — which is hard to use in code. Structured output forces the model to respond in a specific format (like JSON) that matches a schema you define. This makes it easy to extract, validate, and use the model's response directly in your application without manually parsing text.



In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="qwen/qwen3-32b")

response = model.invoke("Hello, from today your name will be MASH.")
response.content

"<think>\nOkay, the user wants me to be called MASH from now on. First, I need to confirm their request and check if there's any context or specific reason behind choosing this name. MASH might be an acronym for something, so maybe they want me to align with a particular identity or function. I should acknowledge their request positively while keeping my role as Qwen clear. I should also ask if there's anything specific they need help with. I need to ensure the response is friendly, open, and encourages them to ask more questions or provide more information if needed. Let me make sure the tone is natural, not too formal, and that the name switch is handled smoothly without confusion. Also, I should avoid any assumptions about what MASH stands for unless they explain it. Keeping it simple and conversational is key here.\n</think>\n\nHello! I'm Qwen, and I'm here to assist you. I understand that you'd like me to use the name MASH for our conversation. Is there a specific reason or contex

### Pydantic

Pydantic is the recommended way to define your output schema in LangChain. You create a class that describes exactly what fields you want in the response, their data types, and optional descriptions — and Pydantic automatically validates that the model's response matches your schema before it reaches your code.

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie (0-10)")


In [9]:
model_pydantic=model.with_structured_output(Movie)
model_pydantic

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027561C44550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027561C44F50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [15]:
model.invoke("Provide me details about movie Avengers:Endgame")

AIMessage(content='<think>\nOkay, so I need to find details about the movie Avengers: Endgame. Let me start by recalling what I know about it. It\'s part of the Marvel Cinematic Universe (MCU), right? I think it was released in 2019. The title suggests it\'s the conclusion of a story arc that started with Avengers: Infinity War. I remember that Thanos is a key character in both movies. \n\nFirst, I should check the director and cast. The directors are the Russo brothers, Anthony and Joe Russo, who also directed Infinity War. The main cast would include the Avengers characters like Iron Man, Thor, Captain America, Black Widow, Hulk, and others. Robert Downey Jr. as Iron Man, Chris Evans as Captain America, Chris Hemsworth as Thor, Mark Ruffalo as Hulk, Scarlett Johansson as Black Widow, Jeremy Renner as Hawkeye. There\'s also Tom Holland as Spider-Man and Paul Rudd as Ant-Man. Thanos is played by Josh Brolin.\n\nThe plot probably centers around the aftermath of Infinity War, where Thano

In [14]:
result=model_pydantic.invoke("Provide me details about movie Avengers:Endgame")
result

Movie(title='Avengers: Endgame', year=2019, genre='Action', director='Anthony Russo, Joe Russo', rating=8.4)

### Message Output alongside parsed strutured

In [18]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie (0-10)")


model_pydantic = model.with_structured_output(Movie, include_raw=True)

result = model_pydantic.invoke("Provide me details about movie Avengers: Endgame")
result

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Avengers: Endgame. Let me see. I need to use the Movie function provided. The required parameters are title, year, genre, director, and rating. \n\nFirst, the title is clearly "Avengers: Endgame". The release year was 2019. The genre would be superhero, maybe action as well. The directors are the Russo brothers, Anthony and Joe Russo. The rating is a bit tricky, but I think it\'s around 8.4 on IMDb. Let me confirm those details.\n\nWait, genre might include action, adventure, and sci-fi. But the main one is superhero. The rating is 8.4, which is a number between 0-10. Okay, I\'ll structure the function call with these parameters. Make sure all required fields are included. Let me double-check the required parameters again: title, year, genre, director, rating. Yes, all set. I\'ll format the JSON accordingly.\n', 'tool_calls': [{'id': 'y6hp4dr3b', 'function': {'arg

### Nested Structured

In [21]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    role: str = Field(description="The role of the actor in the movie")

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    cast: list[Actor] = Field(description="The cast of the movie")
    year: int = Field(description="The release year of the movie")
    genre: list = Field(description="The genre of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie (0-10)")


model_pydantic = model.with_structured_output(Movie)

result = model_pydantic.invoke("Provide me details about movie Avengers: Endgame")
result

Movie(title='Avengers: Endgame', cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Jeremy Renner', role='Clint Barton / Hawkeye'), Actor(name='Tom Holland', role='Peter Parker / Spider-Man'), Actor(name='Paul Rudd', role='Scott Lang / Ant-Man'), Actor(name='Brie Larson', role='Carol Danvers / Captain Marvel')], year=2019, genre=['Action', 'Science Fiction', 'Superhero'], director='Joe Russo, Anthony Russo', rating=8.4)

### TypedDict

TypedDict is a simpler, lightweight alternative to Pydantic for defining structured output. It uses Python's built-in `typing` module — so no extra libraries needed. The key difference from Pydantic is that TypedDict does **not** validate the data at runtime — it only provides type hints for your IDE and code readability. Use it when you want a quick, simple structure without the overhead of full validation.

In [23]:
from typing import TypedDict, Annotated

class Movie(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The release year of the movie"]
    genre: Annotated[list[str], "The genre of the movie"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie (0-10)"]

model_typedict = model.with_structured_output(Movie)
result=model_typedict.invoke("Provide me details about movie Avengers: Endgame")
result

{'director': 'Anthony Russo, Joe Russo',
 'genre': ['Action', 'Sci-Fi', 'Adventure'],
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [ ]:
# Details about the model's profile
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Dataclasses

Dataclasses are Python's built-in way to create simple classes that hold data — no extra libraries needed. They are similar to TypedDict but give you an actual class object instead of a dictionary, which means you can add methods, set default values, and use dot notation to access fields (e.g. `movie.title` instead of `movie["title"]`). Like TypedDict, they do **not** validate data at runtime — they just define the structure. Use them when you want the simplicity of TypedDict but prefer working with objects instead of dictionaries.